# APT Relocation - Not Relocated Scoping

Find APTs that were scoped for relocation in the BFP-based DC input CSVs but were never actually marked as relocated in the layer-14533 snapshot. These are the APTs that the relocation pipeline missed (or rejected).

## Steps

1. **Scoped APT IDs (input)** — read every file ending in `BUILDING_FOOTPRINT.csv` from the relocation-input folder and union all the APT IDs into a single set. This is the universe of APTs that were *supposed* to be relocated.
2. **Relocated APT IDs (actual)** — query the delta table `data_recovery.layer_14533`, filtered to `country = 'USA'` and the metadata tag `metadata:apa:improvement = 'yes'`. This is the set of APTs that were *actually* relocated.
3. **Not-relocated (gap)** — left-anti join: APT IDs from step 1 that are not present in step 2. These are the APTs that should have been relocated but were not.

## 1. Scoped APT IDs from `*BUILDING_FOOTPRINT.csv` (`%scala`)

Read every CSV ending in `BUILDING_FOOTPRINT.csv` from both input folders (one flat, one in the `BUILDING_FOOTPRINT_23_March/` subdirectory), union them, and keep the distinct APT IDs. Adjust `APT_ID_COL` if your CSV column is named differently (e.g. `sourceAptId`, `apt_id`). The resulting `scopedDF` stays in the shared Scala REPL so §3 can use it directly.

In [ ]:
%scala
val INPUT_FOLDERS = Seq(
  "/mnt/aqua/data-correctness/correction-input/SEACO-5792-USA-DC-Execution/PROD-USA-Relocation-DC-Input",
  "/mnt/aqua/data-correctness/correction-input/SEACO-5792-USA-DC-Execution/PROD-USA-Relocation-DC-Input/BUILDING_FOOTPRINT_23_March"
)
val APT_ID_COL = "aptId"   // column in the CSV that holds the APT id

// Non-recursive glob per folder so files in the subdir are not double-read.
val csvPaths: Seq[String] = INPUT_FOLDERS.map(p => s"$p/*BUILDING_FOOTPRINT.csv")

val scopedDF = spark.read
  .option("header", "true")
  .csv(csvPaths: _*)
  .select(APT_ID_COL)
  .withColumnRenamed(APT_ID_COL, "aptId")
  .where("aptId IS NOT NULL")
  .distinct()

val scopedCount = scopedDF.count()
println(s"Reading from ${INPUT_FOLDERS.size} folder(s):")
csvPaths.foreach(p => println(s"  $p"))
println(s"Scoped distinct APT IDs from BUILDING_FOOTPRINT.csv files: $scopedCount")
scopedDF.show(5, truncate = false)

## 2. Relocated APT IDs from layer 14533 (`%scala`)

Load the APT layer-14533 dataset via TomTom's `OrbisElementRepository` (Scala), filter rows where the `tags` array contains an entry with `tagKey.key = "metadata:apa:improvement"` and `value = "yes"`, then attach a `country` column extracted from the `metadata:country` tag (only set when its value is `"USA"`, null otherwise). The final filter keeps USA rows only.

The Scala DataFrame is registered as the temp view `relocated_apt_view` so the Python join in §3 can read it.

In [ ]:
%scala
import org.apache.spark.sql.functions._
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository

val aptDS = new OrbisElementRepository("14533").readAll

val countryTagKey = "metadata:country"

val relocated_apt = aptDS
  // Filter: metadata:apa:improvement = yes
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:apa:improvement"
    && t.getField("value") === "yes"
  ))
  // Extract country value from tags array (USA-only -> column = "USA", else null)
  .withColumn("country",
    filter(col("tags"), t =>
      t.getField("tagKey").getField("key") === countryTagKey && t.getField("value") === "USA"
    ).getItem(0).getField("value")
  )
  // Keep USA rows only
  .filter(col("country") === "USA")

// Expose to Python via temp view for the leftanti join in section 3
relocated_apt.createOrReplaceTempView("relocated_apt_view")

println(s"Relocated USA APTs (metadata:apa:improvement = yes): ${relocated_apt.count()}")
relocated_apt.show(5, truncate = false)

## 3. Not-relocated APT IDs (scoped ∩ ¬relocated) (`%scala`)

Left-anti join: APT IDs from `scopedDF` (§1) that are missing from `relocated_apt` (§2). These are the APTs that were scoped for relocation but never landed the `metadata:apa:improvement = yes` tag. All §1–§3 cells share the same Scala REPL, so `scopedDF` and `scopedCount` from §1 are reused directly.

In [ ]:
%scala
// RELOCATED_ID_COL: the column in `relocated_apt` that holds the APT id.
// OrbisElementRepository usually exposes it as `id`; adjust if your dataset uses a different name.
val RELOCATED_ID_COL = "id"

val relocatedDF = relocated_apt
  .select(RELOCATED_ID_COL)
  .withColumnRenamed(RELOCATED_ID_COL, "aptId")
  .where("aptId IS NOT NULL")
  .distinct()

val relocatedCount = relocatedDF.count()

val notRelocatedDF = scopedDF.join(relocatedDF, Seq("aptId"), "leftanti")

val notRelocatedCount = notRelocatedDF.count()
println(s"Scoped but NOT relocated: $notRelocatedCount APT IDs")
println(s"(scoped=$scopedCount, relocated=$relocatedCount, gap=$notRelocatedCount)")

notRelocatedDF.show(20, truncate = false)